In [1]:
## Vloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

In [2]:
import sys
import os

# Acesso aos módulos do diretório
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: C:\pod\hackathon_pod_2025


##### Carregando pacotes

In [3]:
# Pacotes de manipulacao

import pandas as pd
import numpy as np
import os

# Pacotes de visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

# Funcoes customizadas
import configs.function_basic as funcoes

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 1.5.3
numpy: 1.24.3


## Carregando databases

#### Book_01

In [4]:
# Carregando book_01
book_01 = pd.read_csv(project_root/'database/processed/book_variaveis_01.csv', sep=',')
print("Book 01 data shape:", book_01.shape)

Book 01 data shape: (1290526, 12)


In [5]:
book_01.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290526 entries, 0 to 1290525
Data columns (total 12 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   SAFRA            1290526 non-null  int64  
 1   FLAG_INSTALACAO  1290526 non-null  int64  
 2   FPD              1290526 non-null  int64  
 3   PROD             1290526 non-null  object 
 4   flag_mig2        1290526 non-null  object 
 5   SCORE_01         1281087 non-null  float64
 6   SCORE_02         1289950 non-null  float64
 7   NUM_CPF          1290526 non-null  object 
 8   SCORE_RATEO      1280511 non-null  float64
 9   SCORE_AVG        1280511 non-null  float64
 10  SCORE_DIFF       1280511 non-null  float64
 11  SCORE_MIN        1290526 non-null  float64
dtypes: float64(6), int64(3), object(3)
memory usage: 118.2+ MB


#### Base Dados Cadastrais

In [6]:
## Carregando todos arquivos em parquet de uma pasta

all_files = [os.path.join(project_root/'database/raw/base_dados_cadastrais/', f) for f in os.listdir(project_root/'database/raw/base_dados_cadastrais/') if f.endswith('.parquet')]
df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]

df_dados_cadastrais = pd.concat(df_list, ignore_index=True)
print('Base Dados Cadastrais data shape:', df_dados_cadastrais.shape)

Base Dados Cadastrais data shape: (3900378, 33)


In [7]:
df_dados_cadastrais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900378 entries, 0 to 3900377
Data columns (total 33 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   NUM_CPF           object
 1   SAFRA             object
 2   FLAG_INSTALACAO   object
 3   FPD               object
 4   PROD              object
 5   flag_mig2         object
 6   STATUSRF          object
 7   DATADENASCIMENTO  object
 8   var_03            object
 9   var_02            object
 10  var_04            object
 11  var_05            object
 12  var_06            object
 13  var_07            object
 14  var_08            object
 15  var_09            object
 16  var_10            object
 17  var_11            object
 18  var_12            object
 19  var_13            object
 20  var_14            object
 21  var_15            object
 22  var_16            object
 23  var_17            object
 24  var_18            object
 25  var_19            object
 26  var_20            object
 27  var_21      

#### Merge dos Datasets

In [8]:
# Primeiro vamos transformar a coluna SAFRA para o mesmo formato de book_01
df_dados_cadastrais['SAFRA'] = df_dados_cadastrais['SAFRA'].astype('int64')

In [9]:
cols_to_drop = [
    col for col in df_dados_cadastrais.columns
    if col in book_01.columns
    and col not in ['SAFRA', 'NUM_CPF']
]

df_dados_cadastrais_clean = df_dados_cadastrais.drop(columns=cols_to_drop)

df_book_02 = pd.merge(
    book_01,
    df_dados_cadastrais_clean,
    how='left',
    on=['SAFRA', 'NUM_CPF']
)

In [10]:
# Sanity check
book_01.shape[0] == df_book_02.shape[0]

True

### Feature Engineer

Primeiro iremos aplicar duas regras de negócio:
- Utilizaremos apenas registros que possuam `STATUSRF` = *REGULAR*
- Apenas CPFs que possuam a partir de 18 anos na data de contratação do plano
    - `SAFRA` - `DATADENASCIMENTO` >= 18

- Criar regiões a partir de `CEP_3_digitos` : 
    - 0–1 → Sudeste
    - 2–3 → Sudeste
    - 4 → Sudeste
    - 5 → Nordeste
    - 6–7 → Centro-Oeste / Norte
    - 8–9 → Sul

In [11]:
# Filtrando CPFs que possuam STATUSRF = REGULAR
df_book_02 = df_book_02[df_book_02['STATUSRF'] == 'REGULAR']

In [ ]:
# Filtrando CPFs que nao possuam 18 anos a partir da coluna DT_NASCIMENTO e SAFRA
#df_book_02['DT_NASCIMENTO'] = pd.to_datetime(df_book_02['DT_NASCIMENTO'], format='%Y-%m-%d', errors='coerce')
#df_book_02['AGE'] = df_book_02['SAFRA'] - df_book_02['DT_NASCIMENTO'].dt.year
#df_book_02 = df_book_02[df_book_02['AGE'] >= 18]

In [12]:
df_book_02['STATUSRF'].value_counts()

REGULAR    1281602
Name: STATUSRF, dtype: int64

Nesta seção iremos lidar com valores ausentes e valores continuos
- Deleção de colunas que possuam o limiar de 70% de valores nulos
- Deleção de colunas com cardinalidade igual a 1

In [13]:
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_11,object,1268342,98.97,4949
1,var_10,object,1267529,98.90,1520
2,var_20,object,1265308,98.73,1
3,var_02,object,1210219,94.43,1663
4,var_22,object,1183461,92.34,1
5,var_14,object,1183461,92.34,23
6,var_13,object,1084565,84.63,1494
7,var_07,object,1067684,83.31,86725
8,var_17,object,1053968,82.24,19
9,var_15,object,1053968,82.24,27


In [14]:
# Deletando colunas que possuam mais que 70% de valores faltantes
threshold = 0.7
df_book_02 = funcoes.drop_columns_high_missing(df_book_02, threshold)

🧹 Colunas removidas (> 70% missing): 15
 - var_02: 94.4%
 - var_06: 80.8%
 - var_07: 83.3%
 - var_08: 80.8%
 - var_10: 98.9%
 - var_11: 99.0%
 - var_13: 84.6%
 - var_14: 92.3%
 - var_15: 82.2%
 - var_16: 82.2%
 - var_17: 82.2%
 - var_18: 80.8%
 - var_20: 98.7%
 - var_22: 92.3%
 - var_23: 82.2%


In [15]:
# Agora iremos deletar colunas que possuem cardinalidade igual a 1
df_book_02 = funcoes.drop_single_cardinality_columns(df_book_02)

🧹 Colunas removidas (cardinalidade = 1): 4
 - FLAG_INSTALACAO
 - PROD
 - flag_mig2
 - STATUSRF


In [16]:
# Conferindo metadados
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_09,object,681482,53.17,15
1,var_19,object,681482,53.17,1
2,var_24,object,493680,38.52,2
3,var_21,object,493680,38.52,1
4,var_12,object,493680,38.52,12956
5,var_25,object,131990,10.30,61
6,var_03,object,82850,6.46,100
7,CEP_3_digitos,object,75035,5.85,901
8,var_05,object,53407,4.17,10
9,SCORE_DIFF,float64,8562,0.67,1158


#### Ajustando os tipos de dados

In [ ]:
# Criando novo dataset
#book_variaveis_01 = df_bureau.copy()

In [ ]:
# Salvando o dataframe em csv
#book_variaveis_01.to_csv(project_root/'database/processed/book_variaveis_01.csv', index=False)